# 05 – SQL analysis
Business questions answered with SQL on the `complaints` table (SQLite).

In [2]:
import sqlite3
import pandas as pd

pd.set_option("display.max_colwidth", None)

conn = sqlite3.connect("../data/processed/complaints.db")

def q(sql):
    return pd.read_sql(sql, conn)
    

## Q1 – How has complaint volume changed over time?

In [2]:
q("""
SELECT
    year,
    COUNT(*) AS complaints
FROM complaints
GROUP BY year
ORDER BY year;
""")

,year,complaints
0,2023,93778
1,2024,106630
2,2025,152520
3,2026,158467


**Insight:** Complaints rose by about 63% from 2023 to 2025. Based on the pace up to 14 November 2026, complaints are projected to reach roughly 182,000 by year-end, approximately 19% higher than in 2025. However, rising complaint volumes do not necessarily indicate poorer service; they may also reflect a growing customer base or increased awareness of the CFPB. This makes it important to examine the types of complaints customers raise and how banks respond. The product-level analysis also reveals that checking and savings account complaints increased by 59% from 2024 to 2025, contributing significantly to the overall rise, while credit card complaints in 2026 have already exceeded the full-year 2025 total, indicating that credit cards are a growing area of concern.


In [3]:
q("""
SELECT
    year,
    product,
    COUNT(*) AS complaints
FROM complaints
GROUP BY year, product
ORDER BY year, product;
""")

,year,product,complaints
0,2023,Checking or savings account,49794
1,2023,Credit card,43984
2,2024,Checking or savings account,52652
3,2024,Credit card,53978
4,2025,Checking or savings account,83747
5,2025,Credit card,68773
6,2026,Checking or savings account,78460
7,2026,Credit card,80007


## Q2 – What are customers complaining about most?

In [6]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints
FROM complaints
WHERE product = 'Checking or savings account'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints
0,Managing an account,149535
1,Closing an account,34623
2,Problem with a lender or other company charging your account,33239
3,Problem caused by your funds being low,23072
4,Opening an account,22430


In [7]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints
FROM complaints
WHERE product = 'Credit card'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints
0,Problem with a purchase shown on your statement,75087
1,"Other features, terms, or problems",30704
2,Getting a credit card,30312
3,Fees or interest,27487
4,Closing your account,19024


In [8]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*)
         FROM complaints
         WHERE product = 'Checking or savings account'),
        2
    ) AS percentage
FROM complaints
WHERE product = 'Checking or savings account'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints,percentage
0,Managing an account,149535,56.50
1,Closing an account,34623,13.08
2,Problem with a lender or other company charging your account,33239,12.56
3,Problem caused by your funds being low,23072,8.72
4,Opening an account,22430,8.48


In [10]:
q("""
SELECT
    issue,
    COUNT(*) AS complaints,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*)
         FROM complaints
         WHERE product = 'Credit card'),
        2
    ) AS percentage
FROM complaints
WHERE product = 'Credit card'
GROUP BY issue
ORDER BY complaints DESC
LIMIT 5;
""")

,issue,complaints,percentage
0,Problem with a purchase shown on your statement,75087,30.43
1,"Other features, terms, or problems",30704,12.44
2,Getting a credit card,30312,12.28
3,Fees or interest,27487,11.14
4,Closing your account,19024,7.71


### Q2 – Drill-down: Managing an account

In [11]:
q("""
SELECT
    sub_issue,
    COUNT(*) AS complaints
FROM complaints
WHERE product = 'Checking or savings account'
AND issue = 'Managing an account'
GROUP BY sub_issue
ORDER BY complaints DESC
LIMIT 5;
""")

,sub_issue,complaints
0,Deposits and withdrawals,53470
1,Problem using a debit or ATM card,34728
2,Funds not handled or disbursed as instructed,17333
3,Banking errors,13856
4,Problem accessing account,11591


**Insight:** Checking and savings account complaints are mainly related to managing accounts, representing 56.5% of complaints for this product. A closer look shows that deposits and withdrawals are the biggest problem, accounting for approximately 36% of account-management complaints (53,470 complaints), followed by debit or ATM card issues with 34,728 complaints. In comparison, credit card complaints are more spread across different categories, with problems related to purchases shown on statements accounting for around 30% of complaints. Banks should prioritise improving deposit and withdrawal processes, resolving debit card issues, and making credit card transactions and billing more transparent to reduce customer complaints.

## Q3 – Which banks get the most complaints, and how often do they give relief?

In [3]:
q("""
PRAGMA table_info(complaints);
""")

,cid,name,type,notnull,dflt_value,pk
0,0,date_received,TEXT,0,None,0
1,1,product,TEXT,0,None,0
2,2,sub_product,TEXT,0,None,0
3,3,issue,TEXT,0,None,0
4,4,sub_issue,TEXT,0,None,0
5,5,company_public_response,TEXT,0,None,0
6,6,company,TEXT,0,None,0
7,7,state,TEXT,0,None,0
8,8,tags,TEXT,0,None,0
9,9,submitted_via,TEXT,0,None,0


In [4]:
q("""
SELECT
    company,
    COUNT(*) AS complaints,
    ROUND(
        100.0 * SUM(got_relief) /
        NULLIF(SUM(is_closed), 0), 2
    ) AS relief_rate,
    ROUND(
        100.0 * SUM(got_monetary_relief) /
        NULLIF(SUM(is_closed), 0), 2
    ) AS monetary_relief_rate
FROM complaints
GROUP BY company
ORDER BY complaints DESC
LIMIT 10;
""")

,company,complaints,relief_rate,monetary_relief_rate
0,JPMORGAN CHASE & CO.,52420,17.91,10.84
1,CAPITAL ONE FINANCIAL CORPORATION,51461,19.34,9.84
2,WELLS FARGO & COMPANY,43776,19.05,12.44
3,"CITIBANK, N.A.",43117,45.01,27.34
4,"BANK OF AMERICA, NATIONAL ASSOCIATION",42088,41.78,35.72
5,SYNCHRONY FINANCIAL,24721,24.77,17.53
6,NAVY FEDERAL CREDIT UNION,21889,18.08,9.89
7,Chime Financial Inc,20409,10.11,7.37
8,AMERICAN EXPRESS COMPANY,18358,23.27,15.36
9,U.S. BANCORP,12342,30.02,11.60


**Insight:** JPMorgan Chase has the highest complaint volume with 52,420 complaints, followed by Capital One with 51,461. However, Citibank (45.0%) and Bank of America (41.8%) have much higher relief rates compared to JPMorgan Chase (17.9%), Capital One (19.3%) and Wells Fargo (19.1%). Bank of America also has the highest monetary relief rate at 35.7%, meaning it provides monetary relief in approximately one out of every three closed complaints. In contrast, Chime has the lowest relief rate at 10.1% and monetary relief rate at 7.4%.

However, complaint volumes are not adjusted for the number of customers, so higher complaint counts do not necessarily mean a bank provides poorer service. Additionally, relief rates can depend on the types of complaints each bank receives, meaning a higher relief rate does not automatically indicate better customer service.

Banks such as JPMorgan Chase, Capital One and Wells Fargo could review whether complaints they close without providing relief are being resolved fairly, using Citibank and Bank of America as benchmarks.

## Q4 – Which issues most often lead to customers getting money back?

In [5]:
q("""
SELECT
    product,
    issue,
    COUNT(*) AS complaints,
    ROUND(
        100.0 * SUM(got_monetary_relief) /
        NULLIF(SUM(is_closed), 0), 2
    ) AS monetary_relief_rate
FROM complaints
GROUP BY product, issue
HAVING COUNT(*) >= 1000
ORDER BY monetary_relief_rate DESC
LIMIT 10;
""")

,product,issue,complaints,monetary_relief_rate
0,Credit card,Fees or interest,27487,32.91
1,Credit card,Problem with a purchase shown on your statement,75087,27.81
2,Credit card,Problem when making payments,16487,20.10
3,Checking or savings account,Problem with a lender or other company charging your account,33239,16.06
4,Credit card,"Advertising and marketing, including promotional offers",11381,15.86
5,Credit card,"Other features, terms, or problems",30704,15.32
6,Checking or savings account,Problem caused by your funds being low,23072,14.60
7,Checking or savings account,Managing an account,149535,13.03
8,Checking or savings account,Closing an account,34623,12.16
9,Checking or savings account,Opening an account,22430,9.03


In [6]:
q("""
SELECT
    company,
    COUNT(*) AS total_complaints,
    SUM(product = 'Credit card') AS credit_card_complaints,
    ROUND(
        100.0 * SUM(product = 'Credit card') / COUNT(*), 1
    ) AS credit_card_share
FROM complaints
WHERE company IN (
    'JPMORGAN CHASE & CO.',
    'CAPITAL ONE FINANCIAL CORPORATION',
    'WELLS FARGO & COMPANY',
    'CITIBANK, N.A.',
    'BANK OF AMERICA, NATIONAL ASSOCIATION'
)
GROUP BY company
ORDER BY credit_card_share DESC;
""")

,company,total_complaints,credit_card_complaints,credit_card_share
0,"CITIBANK, N.A.",43117,30984,71.9
1,CAPITAL ONE FINANCIAL CORPORATION,51461,35806,69.6
2,JPMORGAN CHASE & CO.,52420,22010,42.0
3,"BANK OF AMERICA, NATIONAL ASSOCIATION",42088,16291,38.7
4,WELLS FARGO & COMPANY,43776,10061,23.0


**Insight:** Credit card complaints about fees or interest have the highest monetary relief rate at 32.9%, followed by problems with purchases shown on statements at 27.8%. This suggests that billing disputes, unexpected charges and purchase-related issues are important areas where customers frequently receive monetary relief.

A comparison with Q3 shows that differences in relief rates cannot be explained simply by the types of complaints banks receive. For example, Citibank and Capital One have nearly the same credit card complaint share (71.9% and 69.6%), but their relief rates are very different (45.0% and 19.3%). Similarly, Bank of America has a lower credit card complaint share than JPMorgan Chase (38.7% compared to 42.0%), yet its relief rate is more than twice as high (41.8% compared to 17.9%). This suggests that how banks handle and resolve complaints plays an important role in determining customer relief.

Banks should prioritise investigating fee-related complaints and purchase disputes, improve billing transparency, and review their complaint resolution processes to reduce customer dissatisfaction and avoidable refund costs.

## Q5 – Are older Americans and servicemembers treated differently?

In [9]:
q("""
SELECT
    tags,
    COUNT(*) AS complaints,
    ROUND(
        100.0 * SUM(got_relief) /
        NULLIF(SUM(is_closed), 0), 2
    ) AS relief_rate,
    ROUND(
        100.0 * SUM(got_monetary_relief) /
        NULLIF(SUM(is_closed), 0), 2
    ) AS monetary_relief_rate,
    ROUND(
        100.0 * SUM(is_timely) /
        COUNT(*), 2
    ) AS timely_rate
FROM complaints
GROUP BY tags
ORDER BY complaints DESC;
""")

,tags,complaints,relief_rate,monetary_relief_rate,timely_rate
0,No tag,420965,23.35,14.84,99.28
1,Servicemember,41016,22.42,13.78,99.41
2,Older American,39464,27.61,20.14,99.22
3,"Older American, Servicemember",9950,26.78,18.39,99.27


In [10]:
q("""
SELECT
    product,
    tags,
    COUNT(*) AS complaints,
    ROUND(
        100.0 * SUM(got_relief) /
        NULLIF(SUM(is_closed), 0), 1
    ) AS relief_rate,
    ROUND(
        100.0 * SUM(got_monetary_relief) /
        NULLIF(SUM(is_closed), 0), 1
    ) AS monetary_relief_rate,
    ROUND(
        100.0 * SUM(is_timely) /
        COUNT(*), 1
    ) AS timely_rate
FROM complaints
GROUP BY product, tags
ORDER BY product, complaints DESC;
""")

,product,tags,complaints,relief_rate,monetary_relief_rate,timely_rate
0,Checking or savings account,No tag,220036,18.5,13.0,99.3
1,Checking or savings account,Servicemember,21936,17.5,11.6,99.5
2,Checking or savings account,Older American,18122,20.5,15.3,99.3
3,Checking or savings account,"Older American, Servicemember",4559,20.6,14.5,99.4
4,Credit card,No tag,200929,28.7,16.9,99.3
5,Credit card,Older American,21342,33.7,24.2,99.1
6,Credit card,Servicemember,19080,28.1,16.3,99.4
7,Credit card,"Older American, Servicemember",5391,32.0,21.7,99.1


**Insight:** The analysis shows that older Americans receive higher overall relief rates (27.6%) and monetary relief rates (20.1%) compared to the baseline group (23.4% and 14.8%). In contrast, servicemembers receive slightly lower overall relief rates (22.4%) and monetary relief rates (13.8%) than the baseline. The product-level comparison confirms that older Americans receive higher relief rates in both checking and savings accounts (20.5% vs. 18.5%) and credit cards (33.7% vs. 28.7%). Servicemembers receive slightly lower relief in both products, but the differences are small, at 1.0 and 0.6 percentage points respectively. Timely response rates remain very similar across all groups, at approximately 99%.

These findings suggest that older Americans consistently receive higher relief, while the small differences for servicemembers are not conclusive evidence of unfair treatment. Banks should continue monitoring complaint outcomes across these protected groups and review any persistent differences to ensure fair and consistent treatment.